# 10 — Continue training the engine-aligned DDQN

Research question: **Does continued training under the corrected engine model outperform applying the rule fixes without retraining?**

This experiment deliberately abandons P2/P3. It starts from the adopted `dueling_engine_aligned` policy, keeps the original 22 features, network, rewards, learning rate, five-step returns, prioritized replay, D4 augmentation, objective bias 1.00 and center bias 0.04. Only the three engine-timing corrections, the terminal-transition deduplication fix, a fresh continuation optimizer/replay, the agreed opponent curriculum, and a conservative continuation exploration schedule are active. The adopted inference directory and submission ZIP are never overwritten.

## Locked training protocol

Training contains 1,000 rounds: five repetitions of an eight-block cycle, with 25 rounds per block. One learner runtime is retained across all blocks, so the policy, target network, optimizer, replay buffer and counters remain continuous. Checkpoints are archived every 100 rounds; checkpoints 0/200/400/600/800/1000 enter formal validation.

The eight blocks are: rule3; feature-is-everything + two rule agents; deep-learning-killer + two rule agents; both senior agents + one rule agent; old `dueling_hunt_center_04` + two rule agents; feature block repeated; old DDQN + killer + rule; senior mixed block repeated. This realizes the agreed emphasis on standard and external opponents while keeping the old same-family DDQN a minority auxiliary opponent.

## Continuation exploration

The historical `environment_steps` counter is preserved for provenance but does not suppress new exploration. Epsilon uses actions since this continuation began: 0.20→0.12 over steps 0–65k, 0.12→0.07 over 65k–195k, 0.07→0.05 over 195k–325k, then 0.05. The schedule is continuous and checkpointed. It avoids both the stale near-minimum epsilon inherited from the old run and a destructive reset to 0.60.

## Evaluation and model selection

Every formal checkpoint is evaluated without learning on the same seeds and three 20-round lineups: `rule_based_agent × 3`; `feature_is_everything + deep_learning_killer + rule_based_agent`; and `dueling_hunt_center_04 + feature_is_everything + rule_based_agent`. Selection uses score while disclosing tied-first rate, rank, margin, coins, crates, kills, suicides, invalid actions, survival and inference time. The 1,000-round checkpoint is not automatically preferred. A fresh holdout is used once only after choosing one checkpoint.

In [1]:
from pathlib import Path
import json, subprocess, sys
from IPython.display import Markdown, display

def show_table(rows, columns):
    def cell(value):
        return f'{value:.3f}' if isinstance(value, float) else str(value)
    header = '| ' + ' | '.join(columns) + ' |'
    divider = '| ' + ' | '.join(['---'] * len(columns)) + ' |'
    body = ['| ' + ' | '.join(cell(row.get(column, '')) for column in columns) + ' |'
            for row in rows]
    display(Markdown('\n'.join([header, divider, *body])))

def project_root(start=Path.cwd()):
    for path in (start.resolve(), *start.resolve().parents):
        if (path / 'main.py').is_file() and (path / 'agent_code').is_dir():
            return path
    raise FileNotFoundError('project root not found')

ROOT = project_root()
PYTHON = Path(sys.executable).resolve()
ARENA = ROOT / 'experiments/runs/engine_aligned_adapt_20260917'
RUNNER = ROOT / 'evaluation/continue_engine_aligned_ddqn.py'
print('root:', ROOT)
print('python:', PYTHON)
print('arena:', ARENA)

root: <PROJECT_ROOT>
python: <PYTHON_EXECUTABLE>
arena: <PROJECT_ROOT>/experiments/runs/engine_aligned_adapt_20260917


In [2]:
# Reuse a prepared arena. Only create it when no previous run directory exists.
protocol_file = ARENA / 'protocol.json'
if protocol_file.is_file():
    print('Using existing prepared arena; prepare skipped.')
elif ARENA.exists():
    raise RuntimeError(f'Arena exists but has no protocol.json; inspect this partial directory: {ARENA}')
else:
    subprocess.run([str(PYTHON), str(RUNNER), 'prepare', '--arena', str(ARENA)], cwd=ROOT, check=True)
protocol = json.loads(protocol_file.read_text())
protocol

Using existing prepared arena; prepare skipped.


{'base_agent': 'dueling_engine_aligned',
 'base_commit': 'f14c005979843675cd2d219381d2df9f0806ce16',
 'block_size': 25,
 'checkpoint_every': 100,
 'checkpoint_zero_sha256': '709cca4351f3bc5893f766b01b8c9335fa299d25331a275ae281b123df31009a',
 'engine_timing_fixes': True,
 'experiment': 'continue_training_engine_aligned_dddqn',
 'exploration': {'counter': 'environment_steps minus continuation start environment_steps',
  'segments': [{'epsilon': [0.2, 0.12], 'steps': [0, 65000]},
   {'epsilon': [0.12, 0.07], 'steps': [65000, 195000]},
   {'epsilon': [0.07, 0.05], 'steps': [195000, 325000]},
   {'epsilon': [0.05, 0.05], 'steps': [325000, None]}]},
 'formal_validation_every': 200,
 'initialization': {'n_step': 'fresh; retained/cleared normally by round callbacks',
  'optimizer': 'fresh AdamW',
  'policy': 'adopted engine-aligned frozen policy',
  'replay': 'fresh prioritized replay; retained across all blocks',
  'target': 'fresh copy of initial policy; retained across all blocks'},
 'p2': 

In [3]:
# Reuse a complete training run. Partial runs cannot be resumed because optimizer/replay state is in memory.
expected_blocks = protocol['total_rounds'] // protocol['block_size']
block_files = [ARENA / 'results' / f'train_block_{block:02d}.json'
               for block in range(1, expected_blocks + 1)]
final_checkpoint = ARENA / 'checkpoints' / f"engine_aligned_adapt_{protocol['total_rounds']:04d}.pt"
training_complete = final_checkpoint.is_file() and all(path.is_file() for path in block_files)
if training_complete:
    print(f'Using cached training result: {expected_blocks} blocks and {final_checkpoint.name}')
    manifest_file = ARENA / 'training_manifest.json'
    if manifest_file.is_file():
        last = json.loads(manifest_file.read_text())[-1]
        columns = ('round_end', 'continuation_steps', 'learner_updates', 'epsilon', 'elapsed_seconds')
        show_table([{key: last[key] for key in columns}], columns)
elif any(path.exists() for path in block_files):
    completed = sum(path.is_file() for path in block_files)
    raise RuntimeError(f'Found a partial run ({completed}/{expected_blocks} blocks). '
                       'This experiment cannot safely resume without its in-memory optimizer/replay state.')
else:
    # One uninterrupted process keeps replay/optimizer/target state across block changes.
    subprocess.run([str(PYTHON), str(RUNNER), 'train', '--arena', str(ARENA)], cwd=ROOT, check=True)

Using cached training result: 40 blocks and engine_aligned_adapt_1000.pt


| round_end | continuation_steps | learner_updates | epsilon | elapsed_seconds |
| --- | --- | --- | --- | --- |
| 1000 | 317075 | 147534 | 0.051 | 6404.692 |

In [4]:
# Reuse each cached evaluation independently; only missing summaries are computed.
validation_rows = []
for rounds in (0, 200, 400, 600, 800, 1000):
    tag = f'{rounds:04d}'
    checkpoint = ARENA / 'checkpoints' / f'engine_aligned_adapt_{tag}.pt'
    summary_file = ARENA / 'results' / f'validation_{tag}_summary.json'
    if summary_file.is_file():
        print(f'Using cached validation result: {summary_file.name}')
    else:
        if not checkpoint.is_file():
            raise FileNotFoundError(f'Missing checkpoint required for validation: {checkpoint}')
        subprocess.run([str(PYTHON), str(RUNNER), 'eval', '--arena', str(ARENA),
                        '--checkpoint', str(checkpoint), '--tag', tag,
                        '--rounds', '20', '--seed', '1027000'], cwd=ROOT, check=True)
    validation_rows.extend({'checkpoint': tag, **row}
                           for row in json.loads(summary_file.read_text()))

# Present the cached or newly generated results in one table.
columns = ('checkpoint', 'lineup', 'mean_score', 'win_rate', 'mean_rank',
           'coins_per_round', 'crates_per_round', 'kills_per_round',
           'suicides_per_round', 'invalid_per_round', 'survival_rate', 'mean_action_ms')
show_table(validation_rows, columns)

Using cached validation result: validation_0000_summary.json
Using cached validation result: validation_0200_summary.json
Using cached validation result: validation_0400_summary.json
Using cached validation result: validation_0600_summary.json
Using cached validation result: validation_0800_summary.json
Using cached validation result: validation_1000_summary.json


| checkpoint | lineup | mean_score | win_rate | mean_rank | coins_per_round | crates_per_round | kills_per_round | suicides_per_round | invalid_per_round | survival_rate | mean_action_ms |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 0000 | rule3 | 5.000 | 0.650 | 1.500 | 3.000 | 30.750 | 0.400 | 0.150 | 2.500 | 0.800 | 2.281 |
| 0000 | senior_mixed | 2.550 | 0.350 | 2.400 | 2.050 | 22.500 | 0.100 | 0.300 | 2.750 | 0.650 | 2.392 |
| 0000 | old_feature_rule | 3.100 | 0.250 | 2.600 | 2.100 | 27.750 | 0.200 | 0.200 | 3.000 | 0.750 | 2.350 |
| 0200 | rule3 | 5.850 | 0.750 | 1.400 | 3.600 | 32.750 | 0.450 | 0.150 | 3.700 | 0.600 | 2.443 |
| 0200 | senior_mixed | 3.700 | 0.350 | 2.150 | 2.450 | 28.500 | 0.250 | 0.050 | 8.800 | 0.750 | 2.479 |
| 0200 | old_feature_rule | 3.700 | 0.450 | 2.150 | 2.200 | 29.200 | 0.300 | 0.050 | 6.250 | 0.750 | 2.566 |
| 0400 | rule3 | 5.350 | 0.500 | 1.650 | 3.600 | 33.900 | 0.350 | 0.200 | 2.650 | 0.750 | 2.420 |
| 0400 | senior_mixed | 3.000 | 0.300 | 2.450 | 2.000 | 26.150 | 0.200 | 0.050 | 6.700 | 0.700 | 2.462 |
| 0400 | old_feature_rule | 2.800 | 0.200 | 2.250 | 2.300 | 29.950 | 0.100 | 0.050 | 5.700 | 0.750 | 2.541 |
| 0600 | rule3 | 4.300 | 0.600 | 1.650 | 3.550 | 33.000 | 0.150 | 0.000 | 2.950 | 0.950 | 2.382 |
| 0600 | senior_mixed | 2.350 | 0.150 | 2.650 | 1.850 | 24.600 | 0.100 | 0.250 | 4.350 | 0.650 | 2.530 |
| 0600 | old_feature_rule | 2.100 | 0.200 | 2.500 | 2.100 | 30.200 | 0.000 | 0.100 | 3.950 | 0.600 | 2.601 |
| 0800 | rule3 | 5.600 | 0.750 | 1.400 | 3.600 | 33.150 | 0.400 | 0.100 | 2.750 | 0.800 | 2.423 |
| 0800 | senior_mixed | 2.500 | 0.200 | 2.450 | 2.250 | 26.500 | 0.050 | 0.200 | 5.850 | 0.650 | 2.528 |
| 0800 | old_feature_rule | 2.250 | 0.150 | 2.650 | 2.000 | 30.750 | 0.050 | 0.150 | 4.300 | 0.700 | 2.561 |
| 1000 | rule3 | 3.650 | 0.500 | 1.850 | 2.900 | 31.900 | 0.150 | 0.050 | 2.700 | 0.750 | 2.421 |
| 1000 | senior_mixed | 3.300 | 0.300 | 2.250 | 1.800 | 23.000 | 0.300 | 0.250 | 3.800 | 0.550 | 2.529 |
| 1000 | old_feature_rule | 2.800 | 0.250 | 2.200 | 2.050 | 28.650 | 0.150 | 0.150 | 3.750 | 0.600 | 2.590 |

## Completed run — 2026-09-17

Training completed all 1,000 rounds in 6,404.7 seconds. It recorded 317,075 deduplicated continuation actions and increased learner updates from 68,516 to 147,534. Final epsilon was 0.0512. All ten 100-round checkpoints and all 40 block records were saved.

Formal paired validation used 60 games per selected checkpoint with identical initial-condition hashes. The originally planned checkpoints plus requested 0100/0300 comparisons are included. Pooled results:

| checkpoint | score | Δ vs 0000 | paired 95% interval | suicides | invalid |
|---|---:|---:|---:|---:|---:|
| 0000 | 3.550 | +0.000 | [0.000, 0.000] | 0.217 | 2.750 |
| 0100 | 3.467 | -0.083 | [-0.967, 0.750] | 0.083 | **7.533** |
| 0200 | **4.417** | **+0.867** | [-0.217, 1.917] | 0.083 | **6.250** |
| 0300 | 3.600 | +0.050 | [-0.984, 1.067] | 0.117 | 5.717 |
| 0400 | 3.717 | +0.167 | [-0.867, 1.167] | 0.100 | 5.017 |
| 0600 | 2.917 | -0.633 | [-1.250, -0.033] | 0.117 | 3.750 |
| 0800 | 3.450 | -0.100 | [-1.067, 0.817] | 0.150 | 4.300 |
| 1000 | 3.250 | -0.300 | [-1.233, 0.617] | 0.150 | 3.417 |

Under the original prespecified gate, `0200` failed the invalid-action guardrail and would not have advanced. After the exact diagnosis below, the user decided that same-turn target contention should remain reported but should not veto the best scoring policy. `0200` was therefore selected and frozen before opening holdout. Full details are in `experiments/runs/engine_aligned_adapt_20260917/REPORT.md`.

## Frozen selection and expanded paired holdout

Checkpoint `0200` was copied byte-for-byte to `agent_code/frozen_engine_aligned_0200` and `experiments/runs/engine_aligned_adapt_20260917/best/engine_aligned_adapt_best_0200.pt` before holdout. Its SHA-256 is `d715cf26d324502258214f5f755d9a58c2d86ce08922e06aa8917467d76e8dd0`. P2/P3 remain disabled, and the frozen directory is inference-only.

The expanded holdout used 150 fresh paired initial conditions: 50 per lineup from seed bases 1037000, 1038000 and 1039000. Initial-condition hashes match between `0000` and `0200`.

| lineup | 0000 | 0200 | Δ | 0200 higher/equal/lower |
|---|---:|---:|---:|---:|
| rule3 | 4.040 | 3.760 | -0.280 | 21/9/20 |
| senior mixed | 2.380 | 2.660 | +0.280 | 21/14/15 |
| old-DDQN mixed | 2.260 | 2.880 | +0.620 | 22/15/13 |
| pooled | 2.893 | **3.100** | **+0.207** | 64/38/48 |

### Pooled metrics

| metric | 0000 | 0200 | 0200 − 0000 |
|---|---:|---:|---:|
| score | 2.893 | **3.100** | **+0.207** |
| tied-first rate | 0.260 | **0.293** | +0.033 |
| mean rank | 2.373 | **2.253** | -0.120 |
| coins/round | 2.160 | **2.467** | +0.307 |
| crates/round | 25.540 | **30.127** | +4.587 |
| kills/round | **0.147** | 0.127 | -0.020 |
| suicides/round | 0.213 | **0.173** | -0.040 |
| invalid/round | **2.727** | 6.167 | +3.440 |
| survival rate | **0.707** | 0.680 | -0.027 |

### Statistical interpretation

The pooled score difference has a stratified paired bootstrap 95% interval of `[-0.247, 0.667]` from 20,000 resamples (seed 20260918). `0200` scored higher in 64 paired games, tied in 38, and scored lower in 48. The holdout direction is positive but uncertain and smaller than the development result. Its gain is concentrated in the two mixed-opponent lineups, while rule3 moves slightly backward.

Invalid actions rise from 2.727 to 6.167 per round. They remain disclosed but accepted because the deterministic diagnosis attributes them to same-turn target contention rather than actions that were illegal in the observed state. The holdout estimates generalization only: it does not reopen checkpoint selection. `0200` remains the frozen best weight without claiming statistically proven superiority.

Reproducibility artifacts: `experiments/runs/engine_aligned_adapt_20260917/best/holdout_protocol.json`, `best/holdout_summary.json`, and the six raw `results/validation_holdout_{0000,0200}_*.json` files.

## Follow-up: source of `0200` invalid actions

A deterministic diagnostic replay reproduced the same 60 games and all 375 invalid actions. Every failure was a same-turn target collision: the destination was empty and both legal/safe masks allowed the move at decision time, but an opponent executed first and moved into that tile. No failures came from walls, crates, bombs, unavailable bombs or mask violations. The baseline had 165 collisions under the same conditions; `0200` had 375 (8.06→19.08 per 1,000 candidate steps). Of the `0200` collisions, 287 occurred after step 200 and none occurred when the candidate was first in the execution permutation.

The engine collects all actions from one snapshot and then executes them in random order. The current 22-feature model treats opponent positions as static, while late-game opponent progress, objective bias/reward and revisit penalties encourage movement into contested corridors. This makes the execution-time collision diagnosis certain, while the contribution of each policy incentive remains an informed interpretation.

## Decision boundary

Checkpoint `0200` is fixed as the selected frozen weight. The holdout has been consumed once and must not be used to select a different checkpoint or tune hyperparameters. Any future continuation should branch from a copy of `frozen_engine_aligned_0200`; the frozen model and holdout records remain immutable references.